In [10]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
import numpy as np

In [11]:
# Constants
RandomSTATE = 1082
TEST_SIZE = 0.2

In [12]:
# Load real data
try:
    df = pd.read_csv("../../Datasets/csv/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [13]:
# Data Preprocessing
# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label and timestamp
drop_cols = ['timestamp', 'label']
X_real = df.drop(columns=drop_cols)
y_real = df['label']

# Encode labels if needed
if y_real.dtype == object or y_real.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_real = le.fit_transform(y_real)

# Split real data for evaluation
X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
    X_real, y_real, test_size=TEST_SIZE, random_state=RandomSTATE, stratify=y_real
)

In [14]:
# Load synthetic (fake) data
try:
    fake_df = pd.read_csv("syntheticSpoofing_gan_data.csv")
except FileNotFoundError:
    print("Error: The file 'synthetic_gan_data.csv' was not found.")
    print("Please generate synthetic data using your GAN first.")
    exit()

In [15]:
print(fake_df.head())
print(fake_df.info())

       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.382592 -0.007543  0.077072  0.958838          0.965158         -0.278365   
1 -0.374153 -0.006804 -0.031590  0.926899          0.921506         -0.081432   
2 -0.399648  0.018459  0.102795  0.874303          1.283649          0.165151   
3 -0.368292 -0.024050 -0.012271  0.855525          0.603715          0.323909   
4 -0.376194  0.005952 -0.016941  0.926641          1.033364         -0.080970   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0         -0.552430          0.079173            2.775046  36.204815  ...   
1         -0.019904          0.023767            3.014798  36.204823  ...   
2         -0.481015         -0.016148            3.017971  36.204834  ...   
3         -0.147935         -0.233260            3.271205  36.204860  ...   
4          0.002899          0.036381            2.993859  36.204823  ...   

   v_z_valid  xy_reset_counter  z_reset_counter  v

In [16]:
if fake_df['label'].dtype == object or fake_df['label'].dtype.name == 'category':
    y_fake = le.transform(fake_df['label'])
else:
    y_fake = fake_df['label'].values.astype(int)

X_fake = fake_df[X_real.columns]


In [17]:
# Build and train XGBoost classifier on fake data
model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=RandomSTATE,
    eval_metric='mae',
)
model.fit(X_fake, y_fake)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mae', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [18]:
# Predict and evaluate on real test data
y_pred = model.predict(X_test_real)
accuracy = accuracy_score(y_test_real, y_pred)
print(f"Model Accuracy on Test Set (trained on fake, tested on real): {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test_real, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_real, y_pred, zero_division=0))

Model Accuracy on Test Set (trained on fake, tested on real): 96.00%
Confusion Matrix:
[[625   0]
 [ 29  71]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       625
           1       1.00      0.71      0.83       100

    accuracy                           0.96       725
   macro avg       0.98      0.85      0.90       725
weighted avg       0.96      0.96      0.96       725

